## Split train-dev dataset

In this notebook, we will split the train dataset into train-dev-test datasets. We will use the train and dev splits to develop the model and test split to report the performance of our model.

In [9]:
import os
import sys
from functools import partial

import numpy as np
import plotly.express as px
from loguru import logger
from pydantic import BaseModel, model_validator
from dotenv import load_dotenv
import pandas as pd

from sqlalchemy import create_engine

sys.path.insert(0, "..")

from src.visualization.setup import FSDSColors
_ = load_dotenv(override=True)

## Controller

In [10]:
class Args(BaseModel):
    testing: bool = False
    experiment_name: str = "003-split-by-time"
    run_name: str = "001-basic-retrievers"
    notebook_persist_dir: str = None
    random_seed: int = 41

    train_num_days: int = 90
    val_num_days: int = 7
    test_num_days: int = 7
    fit_num_days: int = None

    # Database credentials
    user: str = None
    password: str = None
    db: str = None
    host: str = None
    port: int = None

    @model_validator(mode="before")
    def load_env_vars(cls, values):
        # Load environment variables if not explicitly set
        values["user"] = values.get("user") or os.getenv("POSTGRES_USER")
        values["password"] = values.get("password") or os.getenv("POSTGRES_PASSWORD")
        values["db"] = values.get("db") or os.getenv("POSTGRES_DB")
        values["host"] = values.get("host") or os.getenv("POSTGRES_HOST")
        values["port"] = values.get("port") or os.getenv("POSTGRES_PORT")
        return values

    def init(self):
        self.notebook_persist_dir = os.path.abspath(f"data/{self.run_name}")

        if self.testing:
            logger.info("testing=True so downsampling the data...")
            self.train_num_days = 7
            self.val_num_days = 1
            self.test_num_days = 1

        self.fit_num_days = self.train_num_days + self.val_num_days + self.test_num_days

        return self


args = Args().init()

print(args.model_dump_json(indent=2))

{
  "testing": false,
  "experiment_name": "003-split-by-time",
  "run_name": "001-basic-retrievers",
  "notebook_persist_dir": "/home/dinhln/Desktop/MLOPS/recsys/HM-ScalableRecs/notebooks/data/001-basic-retrievers",
  "random_seed": 41,
  "train_num_days": 90,
  "val_num_days": 7,
  "test_num_days": 7,
  "fit_num_days": 104,
  "user": "lastfirstkiss",
  "password": "nightchange",
  "db": "hm-recsys",
  "host": "localhost",
  "port": 5432
}


### Load data from OLTP